# SERP Heatmap

> Using multiple SERP datasets, create a heatmap for each domain, showing how often it appeared on each position of the SERPs, with some summary statistics.

In [ ]:
# | default_exp serp_heatmap

In [ ]:
# | hide
from nbdev.showdoc import *

In [ ]:
# | echo: false
# from IPython.display import HTML
# import json

# jsonld_dict = {
#   "@context": "https://schema.org",
#   "@type": "HowTo",
#   "name": "Analyze SERPs on a large scale",
# }
# js = f'<script type="application/ld+json">{json.dumps(jsonld_dict)}</script>'
# print(js)

In [ ]:
# | export
import inspect

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.max_columns = None

In [ ]:
# | export
def _concat(text, col_size=20):
    items = sorted(text)
    ncols = -(-len(items) // col_size)  # ceil division
    if ncols <= 1:
        return "<br>".join(items)
    pad = "\u00a0"
    width = max(len(s) for s in items)
    columns = [items[i * col_size : (i + 1) * col_size] for i in range(ncols)]
    nrows = max(len(col) for col in columns)
    lines = []
    for r in range(nrows):
        cells = [
            (col[r] if r < len(col) else "").replace(" ", pad).ljust(width, pad)
            for col in columns
        ]
        lines.append((pad * 3).join(cells))
    return "<br>".join(lines)

In [ ]:
# | export
def _serp_heatmap_panel(serp_df, queries_col, domains_col, ranks_col, num_domains=10):
    """Build a single (non-faceted) SERP heatmap ``go.Figure``.

    This contains the core charting logic and is used both for the standalone
    chart and for each cell of a faceted grid. Layout options that apply to the
    whole figure (title, template, height, width, etc.) are set by the public
    ``serp_heatmap`` wrapper.
    """
    df = serp_df[[queries_col, ranks_col, domains_col]].rename(
        columns={
            queries_col: "searchTerms",
            ranks_col: "rank",
            domains_col: "displayLink",
        }
    )
    df["displayLink"] = df["displayLink"].str.replace(r"^www\.", "", regex=True)
    top_domains = df["displayLink"].value_counts()[:num_domains].index.tolist()
    top_df = df[df["displayLink"].isin(top_domains) & df["displayLink"].ne("")]
    top_df_counts_means = top_df.groupby("displayLink", as_index=False).agg(
        {"rank": ["count", "mean"]}
    )
    top_df_counts_means.columns = ["displayLink", "rank_count", "avg_pos"]
    top_df = pd.merge(top_df, top_df_counts_means).sort_values(
        ["rank_count", "avg_pos"], ascending=[False, True]
    )
    hovertxt_df = top_df.groupby(["rank", "displayLink"], as_index=False).agg(
        {"searchTerms": _concat}
    )
    rank_counts = (
        top_df.groupby(["displayLink", "rank"]).agg({"rank": ["count"]}).reset_index()
    )
    rank_counts.columns = ["displayLink", "rank", "count"]
    num_queries = df["searchTerms"].nunique()
    fig = go.Figure()
    fig.add_scatter(
        x=top_df["displayLink"],
        y=top_df["rank"],
        mode="markers",
        hovertext=top_df["searchTerms"],
        marker={"size": 30, "opacity": 1 / rank_counts["count"].max()},
    )
    fig.add_scatter(
        x=rank_counts["displayLink"],
        y=rank_counts["rank"],
        mode="text",
        text=rank_counts["count"],
    )
    for domain in rank_counts["displayLink"].unique():
        rank_counts_subset = rank_counts[rank_counts["displayLink"] == domain]
        fig.add_scatter(
            x=[domain],
            y=[0],
            mode="text",
            marker={"size": 50},
            text=str(rank_counts_subset["count"].sum()),
        )
        fig.add_scatter(
            x=[domain],
            y=[-1],
            mode="text",
            text=format(rank_counts_subset["count"].sum() / num_queries, ".1%"),
        )
        fig.add_scatter(
            x=[domain],
            y=[-2],
            mode="text",
            marker={"size": 50},
            text=str(
                round(
                    rank_counts_subset["rank"].mul(rank_counts_subset["count"]).sum()
                    / rank_counts_subset["count"].sum(),
                    1,
                )
            ),
        )
    fig.add_scatter(
        x=hovertxt_df["displayLink"],
        y=hovertxt_df["rank"],
        name="",
        hovertemplate="<b>%{x}</b><br>%{hovertext}",
        hoverlabel={"bgcolor": "#efefef", "font": {"family": "monospace"}},
        hovertext=hovertxt_df["searchTerms"],
        mode="markers",
        marker={"opacity": 0, "size": 30},
    )
    minrank, maxrank = (
        int(min(top_df["rank"].unique())),
        int(max(top_df["rank"].unique())),
    )
    fig.layout.yaxis.tickvals = [-2, -1, 0] + list(range(minrank, maxrank + 1))
    fig.layout.yaxis.ticktext = [
        "Avg. Pos.",
        "Coverage",
        "Total<br>appearances",
    ] + list(range(minrank, maxrank + 1))
    fig.layout.yaxis.title = "SERP Rank<br>(number of appearances)"
    fig.layout.showlegend = False
    fig.layout.margin.r = 2
    fig.layout.margin.l = 120
    fig.layout.margin.pad = 0
    fig.layout.yaxis.autorange = "reversed"
    fig.layout.yaxis.zeroline = False
    fig.layout.xaxis.showgrid = False
    fig.layout.yaxis.ticks = "inside"
    fig.layout.xaxis.ticks = "inside"
    fig.layout.yaxis.griddash = "dot"
    return fig


def serp_heatmap(
    serp_df,  # A tidy SERP `DataFrame`: one row per query-domain-rank (e.g. from `advertools.serp_goog`). Duplicate rows show up as the same query repeated in the hover tooltip.
    queries_col,  # Name of the column holding the search queries/keywords.
    domains_col,  # Name of the column holding the domains (display links).
    ranks_col,  # Name of the column holding the rank/position of each result.
    num_domains=10,  # Number of domains to display in the chart.
    height=650,  # Height in pixels. When faceting, this is the height per facet row.
    width=None,  # Width in pixels of the chart.
    title="SERP Heatmap",  # Title of the chart.
    subtitle=None,  # Subtitle of the chart.
    template="none",  # Plotly template to apply.
    facet_row=None,  # Categorical column to split the chart into one row per unique value.
    facet_col=None,  # Categorical column to split into one column per unique value (combine with `facet_row` for a 2D grid).
    **kwargs,  # Valid `make_subplots` params are forwarded to it when faceting; all others go to `fig.update_layout`.
) -> go.Figure:  # A Plotly heatmap figure.
    "Create a heatmap for visualizing domain positions on SERPs."
    required_cols = [queries_col, domains_col, ranks_col]
    facet_cols = [c for c in (facet_row, facet_col) if c is not None]
    missing = [c for c in required_cols + facet_cols if c not in serp_df.columns]
    if missing:
        raise ValueError(
            f"Column(s) not found in serp_df: {missing}. "
            f"Available columns: {serp_df.columns.tolist()}"
        )

    # Route kwargs: make_subplots params vs. layout params.
    reserved = {"rows", "cols", "figure", "subplot_titles", "specs"}
    subplot_param_names = set(inspect.signature(make_subplots).parameters) - reserved
    subplot_kwargs = {
        key: kwargs.pop(key) for key in list(kwargs) if key in subplot_param_names
    }

    def _apply_global_layout(fig, total_height):
        fig.layout.showlegend = False
        fig.layout.template = template
        fig.layout.title = title
        fig.layout.height = total_height
        fig.layout.width = width
        if subtitle is not None:
            fig.layout.title.subtitle.text = subtitle
        if kwargs:
            fig.update_layout(**kwargs)
        return fig

    if not facet_cols:
        fig = _serp_heatmap_panel(
            serp_df, queries_col, domains_col, ranks_col, num_domains
        )
        return _apply_global_layout(fig, height)

    row_vals = serp_df[facet_row].dropna().unique().tolist() if facet_row else [None]
    col_vals = serp_df[facet_col].dropna().unique().tolist() if facet_col else [None]
    nrows, ncols = len(row_vals), len(col_vals)

    def _facet_title(row_val, col_val):
        parts = []
        if facet_row:
            parts.append(f"{facet_row}: {row_val}")
        if facet_col:
            parts.append(f"{facet_col}: {col_val}")
        return ", ".join(parts)

    subplot_titles = [_facet_title(rv, cv) for rv in row_vals for cv in col_vals]
    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=subplot_titles,
        **subplot_kwargs,
    )
    for r, row_val in enumerate(row_vals, start=1):
        for c, col_val in enumerate(col_vals, start=1):
            subset = serp_df
            if facet_row:
                subset = subset[subset[facet_row] == row_val]
            if facet_col:
                subset = subset[subset[facet_col] == col_val]
            if subset.empty:
                continue
            panel = _serp_heatmap_panel(
                subset, queries_col, domains_col, ranks_col, num_domains
            )
            fig.add_traces(panel.data, rows=r, cols=c)
            fig.update_yaxes(
                tickvals=panel.layout.yaxis.tickvals,
                ticktext=panel.layout.yaxis.ticktext,
                title="SERP Rank<br>(number of appearances)" if c == 1 else None,
                autorange="reversed",
                zeroline=False,
                griddash="dot",
                ticks="inside",
                row=r,
                col=c,
            )
            fig.update_xaxes(showgrid=False, ticks="inside", row=r, col=c)

    return _apply_global_layout(fig, height * nrows)

## Exploring the SERP dataset

This table is the result of using the [`advertools.serp_goog`](https://advertools.readthedocs.io/en/master/advertools.serp.html) function, which connects to the official Google Custom Search API. You can use other SERP data providers if you want.

In [ ]:
# | echo: false
serp = pd.read_csv("data/serp_crypto.csv", low_memory=False)
serp.iloc[:5, :120]

,gl,searchTerms,rank,title,snippet,displayLink,link,queryTime,totalResults,cacheId,count,cseName,cx,fileFormat,formattedSearchTime,formattedTotalResults,formattedUrl,htmlFormattedUrl,htmlSnippet,htmlTitle,inputEncoding,kind,mime,outputEncoding,pagemap,safe,searchTime,startIndex,cse_thumbnail,metatags,cse_image,thumbnail,listitem,table,organization,postaladdress,hcard,imageobject,person,interactioncounter,webpage,blogposting,hatomfeed,product,aggregaterating,hproduct,videoobject,collection,creativework,socialmediaposting,softwaresourcecode,comment,article,hreviewaggregate,sitenavigationelement,newsarticle,wpheader,term-def.xml,speakablespecification,maincontentofpage,question,answer,WebPage,corporation,contactpoint,ngo,BreadcrumbList,rating,offer,brand,Document,document,xfn,propertyvalue,review,individualproduct,breadcrumb,webapplication,scraped,og:image,viewport,twitter:card,msapplication-square70x70logo,sailthru.tags,og:site_name,emailt1,msapplication-wide310x150logo,msapplication-tileimage,og:description,twitter:image,twitter:site,msapplication-square310x310logo,sailthru.title,emailcontenttype,msapplication-tilecolor,sailthru.date,og:type,twitter:title,emailvertical,og:title,msapplication-square150x150logo,sailthru.image.thumb,fb:app_id,twitter:description,og:url,sailthru.author,p:domain_verify,metatag_thumbnail,author,twitter:creator,news_keywords,og:locale,twitter:url,fb:admins,twitter:app:id:googleplay,twitter:app:id:ipad,twitter:app:id:iphone,referrer,format-detection,og:image:width
0,us,what is bitcoin,1,Bitcoin - Open source P2P money,Bitcoin uses peer-to-peer technology to operat...,bitcoin.org,https://bitcoin.org/,2022-04-24 18:10:18.884769+00:00,8100000000,eftmar-x2ocJ,10,PySearch,012859022920491477448:pubdbfjmmec,NaN,0.31,"8,100,000,000",https://bitcoin.org/,https://<b>bitcoin</b>.org/,<b>Bitcoin</b> uses peer-to-peer technology to...,<b>Bitcoin</b> - Open source P2P money,utf8,customsearch#result,NaN,utf8,{'cse_thumbnail': [{'src': 'https://encrypted-...,off,0.309795,1,[{'src': 'https://encrypted-tbn1.gstatic.com/i...,[{'og:image': 'https://bitcoin.org/img/icons/o...,[{'src': 'https://bitcoin.org/img/icons/opengr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://bitcoin.org/img/icons/opengraph.png?16...,"width=device-width, initial-scale=1.0, user-sc...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,us,what is bitcoin,2,Bitcoin Definition,Bitcoin is a digital or virtual currency creat...,www.investopedia.com,https://www.investopedia.com/terms/b/bitcoin.asp,2022-04-24 18:10:18.884769+00:00,8100000000,q3G-51w_tGYJ,10,PySearch,012859022920491477448:pubdbfjmmec,NaN,0.31,"8,100,000,000",https://www.investopedia.com/terms/b/bitcoin.asp,https://www.investopedia.com/terms/b/<b>bitcoi...,<b>Bitcoin</b> is a digital or virtual currenc...,<b>Bitcoin</b> Definition,utf8,customsearch#result,NaN,utf8,{'cse_thumbnail': [{'src': 'https://encrypted-...,off,0.309795,1,[{'src': 'https://encrypted-tbn2.gstatic.com/i...,[{'og:image': 'https://www.investopedia.com/th...,[{'src': 'https://www.investopedia.com/thmb/wj...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.investopedia.com/thmb/wjXQJxSVbGQ5...,"width=device-width, initial-scale=1.0",summary,/static/1.229.0/icons/favicons/mstile-70x70.png,"investing,cryptocurrency,bitcoin",Investopedia,Investing,/static/1.229.0/icons/favicons/mstile-310x150.png,/static/1.229.0/icons/favicons/mstile-144x144.png,Bitcoin is a digital or virtual currency creat...,https://www.investopedia.com/thmb/wjXQJxSVbGQ5...,@Investopedia,/static/1.229.0/icons/favicons/mstile-310x310.png,Bitcoin D

#### Watch a discussion about analyzing SERPs:

<iframe width="560" height="315" src="https://www.youtube.com/embed/EX8juHjJ1Bs?si=LfjBraLdOXwAB2Cc" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

This SERP dataset was created by running the [`adv.serp_goog`](https://advertools.readthedocs.io/en/master/advertools.serp.html) function as follows: 

1. Get the top 36 crypto currencies' names
2. Generate two variations for each coin: "what is \<coin>" and "\<coin> price"
3. Run the `serp_goog` function for all coins, query variations and in two countries, US, and UK

#### 36 coins x 2 variations x 2 countries = 144 requests (1,440 SERP results)

As you can see, there are numerous columns that are returned by the API, although many of them are empty. The default ones will always be full though, and they are placed in the beginning of the DataFrame.

### Sample results for the "what is \<coin>" query

In [ ]:
# | echo: false
serp.groupby(["gl", "searchTerms"]).head(3).head(15)[
    ["gl", "searchTerms", "rank", "displayLink", "link", "snippet"]
]

,gl,searchTerms,rank,displayLink,link,snippet
0,us,what is bitcoin,1,bitcoin.org,https://bitcoin.org/,Bitcoin uses peer-to-peer technology to operat...
1,us,what is bitcoin,2,www.investopedia.com,https://www.investopedia.com/terms/b/bitcoin.asp,Bitcoin is a digital or virtual currency creat...
2,us,what is bitcoin,3,www.newscientist.com,https://www.newscientist.com/definition/bitcoin/,Bitcoin is a digital currency which operates f...
10,uk,what is bitcoin,1,bitcoin.org,https://bitcoin.org/,Bitcoin uses peer-to-peer technology to operat...
11,uk,what is bitcoin,2,www.newscientist.com,https://www.newscientist.com/definition/bitcoin/,Bitcoin is a digital currency which operates f...
12,uk,what is bitcoin,3,www.investopedia.com,https://www.investopedia.com/terms/b/bitcoin.asp,Bitcoin is a digital or virtual currency creat...
20,us,what is litecoin,1,litecoin.org,https://litecoin.org/,What is Litecoin? ... Litecoin is a peer-to-pe...
21,us,what is litecoin,2,www.investopedia.com,https://www.investopedia.com/articles/investin...,Litecoin (LTC) is a cryptocurrency created fro...
22,us,what is litecoin,3,coinmarketcap.com,https://coinmarketcap.com/currencies/litecoin/,The live Litecoin price today is $104.96 USD w...
30,uk,what is litecoin,1,www.investopedia.com,https://www.investopedia.com/articles/investin...,Litecoin (LTC) is a cryptocurrency created fro...


### Sample results for the "\<coin> price" query

In [ ]:
# | echo: false
serp[serp["searchTerms"].str.endswith("price")].groupby(["gl", "searchTerms"]).head(
    3
).head(15)[["gl", "searchTerms", "rank", "displayLink", "link", "snippet"]]

,gl,searchTerms,rank,displayLink,link,snippet
720,us,bitcoin price,1,www.coindesk.com,https://www.coindesk.com/price/bitcoin/,"The Bitcoin price is $39,934.91, a change of -..."
721,us,bitcoin price,2,www.coinbase.com,https://www.coinbase.com/price/bitcoin,"April 23, 2022 - The current price of Bitcoin ..."
722,us,bitcoin price,3,coinmarketcap.com,https://coinmarketcap.com/currencies/bitcoin/,"The live Bitcoin price today is $39,717.61 USD..."
730,uk,bitcoin price,1,www.coindesk.com,https://www.coindesk.com/price/bitcoin/,"The Bitcoin price is $39,934.91, a change of -..."
731,uk,bitcoin price,2,www.coinbase.com,https://www.coinbase.com/price/bitcoin,"April 23, 2022 - The current price of Bitcoin ..."
732,uk,bitcoin price,3,coinmarketcap.com,https://coinmarketcap.com/currencies/bitcoin/,"The live Bitcoin price today is $39,606.70 USD..."
740,us,litecoin price,1,coinmarketcap.com,https://coinmarketcap.com/currencies/litecoin/,The live Litecoin price today is $104.96 USD w...
741,us,litecoin price,2,www.coinbase.com,https://www.coinbase.com/price/litecoin,The price of Litecoin has fallen by 7.35% in t...
742,us,litecoin price,3,www.coindesk.com,https://www.coindesk.com/price/litecoin/,Litecoin Price ; 24H Open. $114.00 ; 24H Chang...
750,uk,litecoin price,1,coinmarketcap.com,https://coinmarketcap.com/currencies/litecoin/,The live Litecoin price today is $104.96 USD w...


We can easily create our hetmap with one command:

In [ ]:
serp_heatmap(
    serp, queries_col="searchTerms", domains_col="displayLink", ranks_col="rank"
)

## Modify some of the default options

In [ ]:
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    num_domains=15,
    title='SERP Heatmap - crypto currencies<br><b>"what is <coin>"</b> and <b>"<coin> price"</b>',
    template="ggplot2",
)

In [ ]:
serp_heatmap(
    serp[serp["gl"].eq("uk")],
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    template="plotly_dark",
    num_domains=7,
    title="SERP Heatmap - UK",
    width=800,
    height=600,
)

In [ ]:
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    template="seaborn",
    num_domains=15,
)

## Subtitle and passing extra options to Plotly

Add a `subtitle`, and pass any other option straight through to the underlying Plotly figure with `**kwargs` (forwarded to `fig.update_layout`).

In [ ]:
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    num_domains=8,
    title="SERP Heatmap - crypto",
    subtitle='"what is <coin>" and "<coin> price" keywords, US & UK',
    template="plotly_white",
    paper_bgcolor="#f7f7f7",
    font_color="#333333",
)

## Faceting: split the chart by a categorical column

Instead of manually subsetting the DataFrame and calling the function again and again, pass `facet_row` and/or `facet_col` to split the chart by any categorical column. Combining both produces a 2D grid, and extra `make_subplots` options (like `vertical_spacing`) are forwarded automatically.

First, add a column that labels each query's template:

In [ ]:
serp["query template"] = (
    serp["searchTerms"]
    .str.contains("price")
    .map({True: "<coin> price", False: "what is <coin>"})
)
serp[["searchTerms", "rank", "displayLink", "gl", "query template"]].head()

,searchTerms,rank,displayLink,gl,query template
0,what is bitcoin,1,bitcoin.org,us,what is <coin>
1,what is bitcoin,2,www.investopedia.com,us,what is <coin>
2,what is bitcoin,3,www.newscientist.com,us,what is <coin>
3,what is bitcoin,4,money.cnn.com,us,what is <coin>
4,what is bitcoin,5,www.coinbase.com,us,what is <coin>


In [ ]:
# One column per country (gl):
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    num_domains=7,
    facet_col="gl",
    title="SERP Heatmap by country",
)

In [ ]:
# One row per query template:
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    num_domains=8,
    facet_row="query template",
    title="SERP Heatmap by query template",
)

In [ ]:
# 2D grid: query template (rows) x country (columns):
serp_heatmap(
    serp,
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    num_domains=6,
    facet_row="query template",
    facet_col="gl",
    vertical_spacing=0.08,
    title="SERP Heatmap - query template x country",
)

Faceting makes it easy to compare subsets side by side — notice how different the dominant domains are across query templates and countries, all without manually splitting the DataFrame.


## Add shapes and annotations for better guiding your users

* Add a rectangle to highight one or more domains
* Make the rectangle colored, or simply color the edges of the rectangle
* Optionally add a label to inform users about domain(s) in this rectangle

In [ ]:
fig = serp_heatmap(
    # "price" keywords in the USA:
    serp[serp["gl"].eq("us") & serp["searchTerms"].str.contains("price")],
    queries_col="searchTerms",
    domains_col="displayLink",
    ranks_col="rank",
    template="plotly_white",
    title='SERP Heatmap <b>"<coin> price"</b> keywords – USA',
    num_domains=13,
)
fig.add_shape(
    type="rect", x0=0.5, x1=1.5, y0=-0.5, y1=10.5, fillcolor="gray", opacity=0.15
)
fig.add_shape(
    type="rect",
    x0=4.5,
    x1=6.5,
    y0=-0.5,
    y1=11.5,
    line={"color": "darkgreen"},
    label={
        "text": "This is a label",
        "font": {"family": "Arial Black"},
        "textposition": "bottom center",
    },
)
fig

## SERP Heatmap chart details

* **Avg. Pos.**: The average position of the domain shown on this column
* **Coverage**: The number of appearances of the domain on the SERP, divided by the total queries. A ratio above 100% can exist, where some domains appear more than once in the same SERP.
* **Total appearances**: the number of times the domain appeared in the SERP (from this dataset).
* Numbers in circles: Show the number of times each domain appeared in its respective position.


## Data from other sources

This chart doesn't need the data from the Google custom search API exclusively. There are other similar data sources like YouTube, SERP API, and SearchAPI for example. The only requirement is that you have a DataFrame with columns for the queries, domains, and ranks. Whatever they are named, you simply pass those names through the `queries_col`, `domains_col`, and `ranks_col` parameters — no renaming required. For example, your queries column might be called `keyword` or `query`, and your domains column might be called `domain`.

### Example: data from an LLM (Claude's `serp_claude`)

The chart isn't limited to Google data. The datasets below were produced with a `serp_claude` function that collects the search results Claude returns for a set of queries. Their columns are named differently (`query`, `domain`, `serp_rank`), so we just pass those names through `queries_col`, `domains_col`, and `ranks_col` — no renaming required.

In [ ]:
hotels = pd.read_csv("data/hotels_serps.csv", low_memory=False)
hotels[["query", "city", "num", "serp_rank", "domain"]].head()

,query,city,num,serp_rank,domain
0,3 star hotels in Paris,Paris,3,1,www.booking.com
1,3 star hotels in Paris,Paris,3,2,us.trip.com
2,3 star hotels in Paris,Paris,3,3,www.parisinsidersguide.com
3,3 star hotels in Paris,Paris,3,4,www.tripadvisor.com
4,3 star hotels in Paris,Paris,3,5,www.top-paris-hotels.com


In [ ]:
serp_heatmap(
    hotels,
    queries_col="query",
    domains_col="domain",
    ranks_col="serp_rank",
    num_domains=10,
    title="Hotel SERPs via Claude",
)

In [ ]:
# Facet by star rating (num):
serp_heatmap(
    hotels,
    queries_col="query",
    domains_col="domain",
    ranks_col="serp_rank",
    num_domains=8,
    facet_col="num",
    title="Hotel SERPs by star rating",
)

## Exporting the chart

Because these charts are interactive Plotly figures, you can save them as a standalone HTML file (readers can hover over the circles to see the queries), or as a static image for reports and social previews.

In [ ]:
fig = serp_heatmap(
    serp, queries_col="searchTerms", domains_col="displayLink", ranks_col="rank"
)

# Interactive HTML — hover over the circles to see the queries:
fig.write_html("data/serp_heatmap.html")

# Static image (PNG) — also used as this page's social preview image:
fig.write_image("data/images/serp_heatmap.png", width=1000, engine="kaleido", scale=2)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()